# Driver & Courier Analysis — ATD Dataset

This notebook conducts a **deep analysis of courier behavior** to identify performance patterns, reliability signals, and operational inefficiencies at the driver level.

## Key Questions
1. How is trip volume distributed across drivers? (power-law vs uniform)
2. Which drivers consistently deliver faster / slower? (ATD reliability)
3. Does experience (trip count) correlate with better ATD?
4. How do courier flows (Motorbike vs others) differ in performance?
5. What are the territory-level courier efficiency patterns?
6. Are there drivers with systematically anomalous ATD (outlier factories)?

---
**Sections**
1. Setup & Data Load
2. Driver Activity Distribution
3. ATD Performance by Driver
4. Driver Reliability Analysis
5. Experience vs ATD Relationship
6. Courier Flow Deep Dive
7. Territory × Courier Performance
8. Time Patterns by Courier Type
9. Anomalous Driver Detection
10. Key Takeaways

---
## 1. Setup & Data Load

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})
SEED = 42

RAW_PATH       = Path('../data/raw/BC_A&A_with_ATD.csv')
FEATURES_PATH  = Path('../data/processed/features.parquet')

In [ ]:
# Load processed features if available, else fall back to raw
if FEATURES_PATH.exists():
    df = pd.read_parquet(FEATURES_PATH)
    print(f'Loaded processed features: {df.shape}')
else:
    dtype_map = {
        'region': 'category', 'territory': 'category',
        'country_name': 'category', 'courier_flow': 'category',
        'geo_archetype': 'category', 'merchant_surface': 'category',
    }
    df = pd.read_csv(
        RAW_PATH, dtype=dtype_map,
        parse_dates=['restaurant_offered_timestamp_utc',
                     'order_final_state_timestamp_local',
                     'eater_request_timestamp_local'],
        na_values=['\\N', 'NULL', 'None', ''],
    )
    df = df[df['ATD'] > 0].copy()
    df['ATD_capped'] = df['ATD'].clip(upper=df['ATD'].quantile(0.99))
    df['hour_local'] = (df['restaurant_offered_timestamp_utc'].dt.hour - 6) % 24
    df['is_weekend'] = df['restaurant_offered_timestamp_utc'].dt.dayofweek.isin([5, 6]).astype(int)
    df['is_peak_hour'] = df['hour_local'].isin(list(range(12, 15)) + list(range(19, 24))).astype(int)
    print(f'Loaded raw data (fallback): {df.shape}')

# Ensure driver_uuid column exists
print(f'Unique drivers: {df["driver_uuid"].nunique():,}')
print(f'Rows with driver_uuid: {df["driver_uuid"].notna().sum():,}')

---
## 2. Driver Activity Distribution

We examine how trips are distributed across drivers — a power-law distribution would mean a small number of drivers handle a large share of volume.

In [ ]:
driver_trips = (
    df[df['driver_uuid'].notna()]
    .groupby('driver_uuid')
    .agg(
        trip_count=('ATD_capped', 'count'),
        atd_median=('ATD_capped', 'median'),
        atd_mean=('ATD_capped', 'mean'),
        atd_std=('ATD_capped', 'std'),
        atd_p25=('ATD_capped', lambda x: x.quantile(0.25)),
        atd_p75=('ATD_capped', lambda x: x.quantile(0.75)),
        total_distance_mean=('total_distance', 'mean') if 'total_distance' in df.columns else ('ATD_capped', 'count'),
    )
    .reset_index()
)

# Coefficient of variation (lower = more reliable)
driver_trips['atd_cv'] = driver_trips['atd_std'] / driver_trips['atd_mean']
driver_trips['atd_iqr'] = driver_trips['atd_p75'] - driver_trips['atd_p25']

print(f'Total unique drivers with trips: {len(driver_trips):,}')
print('\nTrip count distribution:')
print(driver_trips['trip_count'].describe().round(1))

In [ ]:
# Pareto analysis: what % of drivers handle what % of trips?
sorted_trips = driver_trips.sort_values('trip_count', ascending=False)
sorted_trips['cumulative_pct_drivers'] = np.arange(1, len(sorted_trips) + 1) / len(sorted_trips) * 100
sorted_trips['cumulative_pct_trips'] = sorted_trips['trip_count'].cumsum() / sorted_trips['trip_count'].sum() * 100

# Find 80/20 point
idx_80 = (sorted_trips['cumulative_pct_trips'] >= 80).idxmax()
pct_drivers_80 = sorted_trips.loc[idx_80, 'cumulative_pct_drivers']

fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Trip count distribution (log scale)
axes[0].hist(driver_trips['trip_count'], bins=80, color='steelblue', edgecolor='none', log=True)
axes[0].set_title('Driver Trip Count Distribution (log y)')
axes[0].set_xlabel('Trips per Driver')
axes[0].set_ylabel('Count (log)')

# Pareto curve
axes[1].plot(
    sorted_trips['cumulative_pct_drivers'],
    sorted_trips['cumulative_pct_trips'],
    color='steelblue', linewidth=2
)
axes[1].axhline(80, color='red', linestyle='--', alpha=0.7, label='80% of trips')
axes[1].axvline(pct_drivers_80, color='orange', linestyle='--', alpha=0.7,
                label=f'Top {pct_drivers_80:.1f}% of drivers')
axes[1].set_title('Pareto: Drivers vs Trip Volume')
axes[1].set_xlabel('Cumulative % of Drivers')
axes[1].set_ylabel('Cumulative % of Trips')
axes[1].legend(fontsize=9)

# Percentile buckets
driver_trips['activity_tier'] = pd.qcut(
    driver_trips['trip_count'],
    q=[0, 0.25, 0.50, 0.75, 0.90, 1.0],
    labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Top 10%'],
    duplicates='drop'
)
tier_vol = driver_trips.groupby('activity_tier', observed=True)['trip_count'].sum()
tier_vol.plot(kind='bar', ax=axes[2], color='coral', edgecolor='none')
axes[2].set_title('Total Trips by Driver Activity Tier')
axes[2].set_ylabel('Total Trips')
axes[2].tick_params(axis='x', rotation=20)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Driver Activity Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nPareto insight: Top {pct_drivers_80:.1f}% of drivers account for 80% of all trips.')

---
## 3. ATD Performance by Driver

We rank drivers by their median ATD and identify top and bottom performers.

In [ ]:
# Focus on drivers with ≥ 20 trips for statistical reliability
MIN_TRIPS = 20
reliable_drivers = driver_trips[driver_trips['trip_count'] >= MIN_TRIPS].copy()
print(f'Drivers with ≥ {MIN_TRIPS} trips: {len(reliable_drivers):,} '
      f'({len(reliable_drivers)/len(driver_trips)*100:.1f}% of all drivers)')
print(f'These drivers account for '
      f'{reliable_drivers["trip_count"].sum()/driver_trips["trip_count"].sum()*100:.1f}% of all trips')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Distribution of driver median ATD
axes[0].hist(reliable_drivers['atd_median'], bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(reliable_drivers['atd_median'].median(), color='red', linestyle='--',
                label=f'Median {reliable_drivers["atd_median"].median():.0f} min')
axes[0].set_title('Distribution of Driver Median ATD')
axes[0].set_xlabel('Driver Median ATD (min)')
axes[0].legend()

# Distribution of driver ATD std
axes[1].hist(reliable_drivers['atd_std'].dropna(), bins=60, color='coral', edgecolor='none')
axes[1].set_title('Distribution of Driver ATD Std Dev')
axes[1].set_xlabel('ATD Std Dev (min)')

# Median ATD vs trip count
sample_d = reliable_drivers.sample(min(5000, len(reliable_drivers)), random_state=SEED)
axes[2].scatter(sample_d['trip_count'], sample_d['atd_median'],
                alpha=0.15, s=8, color='mediumpurple')
axes[2].set_title('Driver Trip Count vs Median ATD')
axes[2].set_xlabel('Trip Count')
axes[2].set_ylabel('Median ATD (min)')

plt.suptitle('Driver ATD Performance Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 fastest and bottom 10 slowest drivers (median ATD, ≥50 trips)
heavy_drivers = reliable_drivers[reliable_drivers['trip_count'] >= 50]

top10_fast = heavy_drivers.nsmallest(10, 'atd_median')[['driver_uuid', 'trip_count', 'atd_median', 'atd_std', 'atd_cv']]
top10_slow = heavy_drivers.nlargest(10, 'atd_median')[['driver_uuid', 'trip_count', 'atd_median', 'atd_std', 'atd_cv']]

print('Top 10 Fastest Drivers (≥ 50 trips, by median ATD):')
display(top10_fast.round(2))

print('\nTop 10 Slowest Drivers (≥ 50 trips, by median ATD):')
display(top10_slow.round(2))

---
## 4. Driver Reliability Analysis

**Reliability** = consistency of delivery times. A driver with low ATD variability (low CV) is predictable, which is key for customer experience.

In [ ]:
# 2×2 matrix: speed (median ATD) vs reliability (CV)
med_median = reliable_drivers['atd_median'].median()
med_cv     = reliable_drivers['atd_cv'].median()

def classify_driver(row):
    fast = row['atd_median'] < med_median
    reliable = row['atd_cv'] < med_cv
    if fast and reliable:     return 'Star (Fast & Reliable)'
    if fast and not reliable: return 'Erratic (Fast but Unpredictable)'
    if not fast and reliable: return 'Steady (Slow but Consistent)'
    return 'Struggling (Slow & Unpredictable)'

reliable_drivers['driver_class'] = reliable_drivers.apply(classify_driver, axis=1)

class_counts = reliable_drivers['driver_class'].value_counts()
print('Driver classification (Speed × Reliability):')
print(class_counts.to_string())
print(f'\nTotal classified: {class_counts.sum():,}')

In [ ]:
COLORS = {
    'Star (Fast & Reliable)':          'seagreen',
    'Erratic (Fast but Unpredictable)': 'gold',
    'Steady (Slow but Consistent)':    'steelblue',
    'Struggling (Slow & Unpredictable)': 'tomato',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: median ATD vs CV
sample_r = reliable_drivers.sample(min(8000, len(reliable_drivers)), random_state=SEED)
for cls, grp in sample_r.groupby('driver_class'):
    axes[0].scatter(grp['atd_median'], grp['atd_cv'],
                    alpha=0.3, s=10, label=cls, color=COLORS.get(cls, 'gray'))

axes[0].axvline(med_median, color='black', linestyle='--', linewidth=0.8)
axes[0].axhline(med_cv, color='black', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Driver Median ATD (min)')
axes[0].set_ylabel('Coefficient of Variation (ATD)')
axes[0].set_title('Driver Speed vs Reliability')
axes[0].legend(fontsize=8, loc='upper right')

# Bar: driver class distribution
class_counts.plot(kind='bar', ax=axes[1],
                  color=[COLORS.get(c, 'gray') for c in class_counts.index],
                  edgecolor='none')
axes[1].set_title('Driver Count by Classification')
axes[1].tick_params(axis='x', rotation=20)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():,.0f}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

plt.suptitle('Driver Reliability Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Experience vs ATD Relationship

Do drivers improve over time (within the dataset window)? We use trip count as a proxy for experience.

In [ ]:
# Bin drivers by trip count quintile
trip_bins = [0, 5, 10, 20, 50, 100, 200, np.inf]
trip_labels = ['1-5', '6-10', '11-20', '21-50', '51-100', '101-200', '200+']

driver_trips['experience_tier'] = pd.cut(
    driver_trips['trip_count'], bins=trip_bins, labels=trip_labels, include_lowest=True
)

exp_stats = (
    driver_trips.groupby('experience_tier', observed=True)
    .agg(
        n_drivers=('driver_uuid', 'count'),
        total_trips=('trip_count', 'sum'),
        median_atd=('atd_median', 'median'),
        median_cv=('atd_cv', 'median'),
    )
    .round(2)
)
display(exp_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

exp_stats['median_atd'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Median ATD by Driver Experience Tier')
axes[0].set_xlabel('Trips per Driver')
axes[0].set_ylabel('Median ATD (min)')
axes[0].tick_params(axis='x', rotation=0)

exp_stats['median_cv'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='none')
axes[1].set_title('Median ATD CV by Driver Experience Tier')
axes[1].set_xlabel('Trips per Driver')
axes[1].set_ylabel('Median CV')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Experience Effect on Delivery Performance', fontweight='bold')
plt.tight_layout()
plt.show()

# Regression line on scatter
sample_exp = driver_trips[(driver_trips['trip_count'] <= 200) &
                           driver_trips['atd_median'].notna()].sample(
    min(10_000, len(driver_trips)), random_state=SEED
)
corr_exp = sample_exp['trip_count'].corr(sample_exp['atd_median'])
print(f'Pearson r (trip_count vs driver median ATD): {corr_exp:.4f}')

---
## 6. Courier Flow Deep Dive

Different courier flows (Motorbike, Car, Bicycle, etc.) have fundamentally different speed, capacity, and distance profiles.

In [ ]:
p99_atd = df['ATD_capped'].quantile(0.99) if 'ATD_capped' in df.columns else df['ATD'].quantile(0.99)
atd_col = 'ATD_capped' if 'ATD_capped' in df.columns else 'ATD'

flow_stats = (
    df.groupby('courier_flow', observed=True)
    .agg(
        trips=(atd_col, 'count'),
        atd_median=(atd_col, 'median'),
        atd_mean=(atd_col, 'mean'),
        atd_p25=(atd_col, lambda x: x.quantile(0.25)),
        atd_p75=(atd_col, lambda x: x.quantile(0.75)),
        atd_p95=(atd_col, lambda x: x.quantile(0.95)),
    )
    .sort_values('atd_median')
)
flow_stats['trip_pct'] = (flow_stats['trips'] / flow_stats['trips'].sum() * 100).round(1)
flow_stats['atd_iqr']  = flow_stats['atd_p75'] - flow_stats['atd_p25']

print('Courier Flow Performance Summary:')
display(flow_stats.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Volume by courier flow
flow_stats['trips'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Trip Volume by Courier Flow')
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Median ATD with IQR
flows = flow_stats.index.tolist()
medians = flow_stats['atd_median'].values
p25s    = flow_stats['atd_p25'].values
p75s    = flow_stats['atd_p75'].values
colors  = ['steelblue' if v <= flow_stats['atd_median'].median() else 'tomato' for v in medians]
bars    = axes[1].bar(flows, medians, color=colors, edgecolor='none')
axes[1].errorbar(flows, medians, yerr=[medians - p25s, p75s - medians],
                 fmt='none', ecolor='black', capsize=4, linewidth=1.5)
axes[1].set_title('Median ATD ± IQR by Courier Flow')
axes[1].set_ylabel('ATD (min)')
axes[1].tick_params(axis='x', rotation=20)

# Box plot on sample
sample_flow = df.sample(min(50_000, len(df)), random_state=SEED)
flow_order  = df.groupby('courier_flow', observed=True)[atd_col].median().sort_values().index
sns.boxplot(
    data=sample_flow, x='courier_flow', y=atd_col, order=flow_order,
    ax=axes[2], palette='muted',
    flierprops=dict(marker='.', markersize=1, alpha=0.3),
)
axes[2].set_title('ATD Distribution by Courier Flow')
axes[2].set_xlabel('')
axes[2].set_ylabel('ATD (min)')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Courier Flow Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distance profile by courier flow
if 'total_distance' in df.columns:
    dist_by_flow = (
        df.groupby('courier_flow', observed=True)['total_distance']
        .agg(['median', 'mean', 'std'])
        .round(2)
        .sort_values('median')
    )
    print('Total distance profile by courier flow:')
    display(dist_by_flow)

    fig, ax = plt.subplots(figsize=(10, 4))
    dist_by_flow['median'].plot(kind='bar', ax=ax, color='teal', edgecolor='none')
    ax.set_title('Median Total Distance by Courier Flow')
    ax.set_ylabel('Total Distance (km)')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()

---
## 7. Territory × Courier Performance

Cross-tab the territory and courier flow to find which combinations drive the highest and lowest ATDs.

In [ ]:
ter_flow = (
    df.groupby(['territory', 'courier_flow'], observed=True)
    .agg(
        trips=(atd_col, 'count'),
        atd_median=(atd_col, 'median'),
        atd_p75=(atd_col, lambda x: x.quantile(0.75)),
    )
    .reset_index()
    .sort_values('atd_median', ascending=False)
)

print('Top 15 worst-performing territory × courier combinations:')
display(ter_flow.head(15).round(2))
print('\nTop 15 best-performing territory × courier combinations:')
display(ter_flow.tail(15).round(2))

In [ ]:
# Pivot heatmap: territory × courier_flow median ATD
heatmap_df = ter_flow.pivot(index='territory', columns='courier_flow', values='atd_median').fillna(0)

fig, ax = plt.subplots(figsize=(14, max(6, len(heatmap_df) * 0.4)))
sns.heatmap(
    heatmap_df, ax=ax, cmap='YlOrRd',
    annot=True, fmt='.0f', cbar_kws={'label': 'Median ATD (min)'},
    linewidths=0.3,
)
ax.set_title('Median ATD: Territory × Courier Flow', fontweight='bold')
ax.set_xlabel('Courier Flow')
ax.set_ylabel('Territory')
plt.tight_layout()
plt.show()

---
## 8. Time Patterns by Courier Type

Does peak-hour performance degradation affect all courier types equally?

In [ ]:
if 'hour_local' not in df.columns:
    df['hour_local'] = (df['restaurant_offered_timestamp_utc'].dt.hour - 6) % 24

time_flow = (
    df.groupby(['hour_local', 'courier_flow'], observed=True)[atd_col]
    .median()
    .unstack('courier_flow')
)

fig, ax = plt.subplots(figsize=(14, 5))
time_flow.plot(ax=ax, linewidth=2, marker='o', markersize=4)
ax.set_title('Median ATD by Hour (Local) & Courier Flow', fontweight='bold')
ax.set_xlabel('Hour of Day (Local)')
ax.set_ylabel('Median ATD (min)')
ax.legend(title='Courier Flow', bbox_to_anchor=(1.01, 1))
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

# Peak vs off-peak delta per courier type
if 'is_peak_hour' in df.columns:
    peak_delta = (
        df.groupby(['courier_flow', 'is_peak_hour'], observed=True)[atd_col]
        .median()
        .unstack('is_peak_hour')
        .rename(columns={0: 'off_peak', 1: 'peak'})
    )
    peak_delta['peak_uplift_min'] = peak_delta['peak'] - peak_delta['off_peak']
    peak_delta['peak_uplift_pct'] = peak_delta['peak_uplift_min'] / peak_delta['off_peak'] * 100
    print('Peak hour ATD uplift by courier flow:')
    display(peak_delta.round(2))

---
## 9. Anomalous Driver Detection

Identify drivers with statistically anomalous ATD that could signal:
- Data quality issues (wrong trip assignments)
- Drivers gaming the system
- Operational problems (poor route knowledge)

In [ ]:
# Z-score of driver median ATD relative to their territory × courier_flow peers
peer_stats = (
    df.groupby(['territory', 'courier_flow'], observed=True)[atd_col]
    .agg(['median', 'std'])
    .rename(columns={'median': 'peer_median', 'std': 'peer_std'})
)

# Merge driver stats with territory data
driver_with_territory = (
    df[df['driver_uuid'].notna()]
    .groupby(['driver_uuid', 'territory', 'courier_flow'], observed=True)[atd_col]
    .agg(['count', 'median'])
    .reset_index()
    .rename(columns={'count': 'trips', 'median': 'driver_median'})
)

# Use mode territory per driver
driver_main_segment = (
    driver_with_territory.sort_values('trips', ascending=False)
    .drop_duplicates('driver_uuid')
)

driver_main_segment = driver_main_segment.merge(
    peer_stats.reset_index(),
    on=['territory', 'courier_flow'],
    how='left'
)
driver_main_segment['peer_z'] = (
    (driver_main_segment['driver_median'] - driver_main_segment['peer_median'])
    / (driver_main_segment['peer_std'] + 1e-6)
)

# Anomalous drivers: |z| > 3 and ≥ 20 trips
anomalous = driver_main_segment[
    (driver_main_segment['peer_z'].abs() > 3) &
    (driver_main_segment['trips'] >= 20)
].copy().sort_values('peer_z', ascending=False)

print(f'Anomalous drivers (|z| > 3, ≥ 20 trips): {len(anomalous):,}')
print(f'  Abnormally slow (z > 3): {(anomalous["peer_z"] > 3).sum():,}')
print(f'  Abnormally fast (z < -3): {(anomalous["peer_z"] < -3).sum():,}')

print('\nTop 10 slowest anomalous drivers:')
display(anomalous.head(10)[['driver_uuid', 'territory', 'courier_flow', 'trips', 'driver_median', 'peer_median', 'peer_z']].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(driver_main_segment['peer_z'].clip(-6, 6), bins=80, color='steelblue', edgecolor='none')
ax.axvline(3, color='red', linestyle='--', label='z = +3 (slow outlier)')
ax.axvline(-3, color='green', linestyle='--', label='z = −3 (fast outlier)')
ax.set_title('Distribution of Driver ATD Z-Score (vs Territory × Courier Peers)', fontweight='bold')
ax.set_xlabel('Z-score (clipped to ±6)')
ax.set_ylabel('Driver Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Key Takeaways

In [ ]:
n_drivers = df['driver_uuid'].nunique()
pareto_pct = round(pct_drivers_80, 1) if 'pct_drivers_80' in dir() else '~20'

summary = {
    'Total unique drivers'                : f'{n_drivers:,}',
    'Drivers with ≥20 trips (reliable)'   : f'{len(reliable_drivers):,}  ({len(reliable_drivers)/n_drivers*100:.1f}%)',
    'Pareto: top X% handle 80% of trips'  : f'Top {pareto_pct}% of drivers',
    'Star drivers (Fast & Reliable)'      : f'{(reliable_drivers["driver_class"]=="Star (Fast & Reliable)").sum():,}' if 'driver_class' in reliable_drivers.columns else 'N/A',
    'Struggling drivers'                  : f'{(reliable_drivers["driver_class"]=="Struggling (Slow & Unpredictable)").sum():,}' if 'driver_class' in reliable_drivers.columns else 'N/A',
    'Anomalous drivers (|z|>3, ≥20 trips)': f'{len(anomalous):,}' if 'anomalous' in dir() else 'N/A',
    'Dominant courier flow'               : str(df['courier_flow'].value_counts().index[0]),
    'Peak hour ATD uplift (Motorbike)'    : f'{peak_delta.loc["Motorbike","peak_uplift_min"]:.1f} min' if 'peak_delta' in dir() and 'Motorbike' in (peak_delta.index if 'peak_delta' in dir() else []) else 'N/A',
}

print('═' * 55)
print('  DRIVER & COURIER ANALYSIS — KEY FINDINGS')
print('═' * 55)
for k, v in summary.items():
    print(f'  {k:<42} : {v}')
print('═' * 55)